# GrieveAI audited MuRIL + LoRA diagnostic

Canonical experiment using checked-in synthetic data only. Metrics are not VCET or real-world performance.

## 1. Fixed configuration
The diagnostic uses seed 42, the duplicate-aware 801/275/247 split, and the same data for the TF-IDF + SVM baseline.

In [5]:
SEED = 42
BATCH_SIZE = 8
MODEL_NAME = "google/muril-base-cased"
DATA_PATH = "data/processed/grievances_synthetic.csv"
TAXONOMY_PATH = "config/taxonomy.json"
SMOKE_OUTPUT_DIR = "/content/GrieveAI/checkpoints/diagnostic_1_epoch"
MAX_LENGTH = 128


## 2. Clone the pushed repository and record its commit

In [6]:
%cd /content/GrieveAI
import subprocess
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert GIT_COMMIT == "f67a1947da6d4b749e0e041150ceb962916c55af"
print("Git commit:", GIT_COMMIT)


/content/GrieveAI
Git commit: f67a1947da6d4b749e0e041150ceb962916c55af


## 3. Install required ML packages, record versions and check GPU

In [7]:
!pip install -q -r requirements-colab.txt
import hashlib, platform, torch, transformers, peft, sklearn, pandas as pd
from pathlib import Path
versions = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "transformers": transformers.__version__, "peft": peft.__version__, "sklearn": sklearn.__version__, "pandas": pd.__version__}
manifest_files = ["src/train.py", "src/model.py", "src/data_splits.py", "train_baselines.py", "config/taxonomy.json", DATA_PATH]
source_manifest = {p: hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in manifest_files}
print({"commit": GIT_COMMIT, "versions": versions, "source_hashes": source_manifest})
assert torch.cuda.is_available(), "Select a GPU runtime before the MuRIL check."
assert MODEL_NAME == __import__("src.model", fromlist=["MURIL_CHECKPOINT"]).MURIL_CHECKPOINT


{'commit': 'f67a1947da6d4b749e0e041150ceb962916c55af', 'versions': {'python': '3.13.15', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'Tesla T4', 'transformers': '5.16.1', 'peft': '0.20.0', 'sklearn': '1.6.1', 'pandas': '2.2.3'}, 'source_hashes': {'src/train.py': 'd8e2efc8e375cedd91de17de145fd80dd8f680774fc59049653bfc9a82b7c699', 'src/model.py': 'fe24db2ad1cf3fcd2e5116313fb0f3356d6e029aee435f8ecff9b2c2e481bf20', 'src/data_splits.py': '7171432d9d0400f7f2683b0134392de394d0be1c9d3a4aff783a071ff76633ff', 'train_baselines.py': '87fd9871e79611e04580177cd0a2b8d93dc7281205ed75760d6fc95a67c67084', 'config/taxonomy.json': '3fcbc9aa5082b3d76f4460e713a86a3f514ec544ee971c73338ef599aa7d5e8f', 'data/processed/grievances_synthetic.csv': '031a31c02f66b38f7f471d421bdb4f5e24a6b2d1234efc706bf51ded0f163396'}}


## 4. Validate taxonomy mapping and frozen split

In [8]:
import json
from src.data_splits import make_splits, validate_taxonomy_labels, has_group_leakage
df = pd.read_csv(DATA_PATH)
taxonomy = json.loads(Path(TAXONOMY_PATH).read_text(encoding="utf-8"))
validate_taxonomy_labels(df, taxonomy)
assert len(df) == 1323 and df.category.nunique() == 7 and df.subcategory.nunique() == 28
assert all(len(v["subcategories"]) == 4 for v in taxonomy["categories"].values())
train_df, val_df, test_df = make_splits(df, seed=SEED)
assert (len(train_df), len(val_df), len(test_df)) == (801, 275, 247)
assert not has_group_leakage((train_df, val_df, test_df))
print({"rows": len(df), "split_sizes": [len(train_df), len(val_df), len(test_df)], "category_counts": df.category.value_counts().sort_index().to_dict(), "taxonomy_mapping": "valid"})


{'rows': 1323, 'split_sizes': [801, 275, 247], 'category_counts': {'Academics': 190, 'Canteen': 190, 'Examinations': 193, 'Fees_Accounts': 186, 'IT_Library': 190, 'Infrastructure': 188, 'Transport': 186}, 'taxonomy_mapping': 'valid'}


## 5. Run the keyword and TF-IDF + SVM baselines on the same split

In [9]:
!python train_baselines.py --seed 42


{
  "dataset_kind": "synthetic_demo_pipeline_validation",
  "research_claim": "not_vcet_pilot_results",
  "split": {
    "train_rows": 801,
    "validation_rows": 275,
    "test_rows": 247,
    "duplicate_group_leakage": false
  },
  "validation": {
    "keyword_rules": {
      "category_macro_f1": 0.6933683378581981,
      "subcategory_macro_f1": 0.581201559165653,
      "priority_mae": 0.7781818181818182,
      "priority_recall_high": 0.0
    },
    "tfidf_svm": {
      "category_macro_f1": 0.8571770635641094,
      "subcategory_macro_f1": 0.7436554904923743,
      "priority_mae": 0.6009919329900256,
      "priority_recall_high": 0.15
    }
  },
  "test": {
    "keyword_rules": {
      "category_macro_f1": 0.6736997184860051,
      "subcategory_macro_f1": 0.6292192762535079,
      "priority_mae": 0.6963562753036437,
      "priority_recall_high": 0.0
    },
    "tfidf_svm": {
      "category_macro_f1": 0.8324908639819526,
      "subcategory_macro_f1": 0.7600331870639994,
      "priori

## 6. Test MuRIL tokenizer output, forward pass, hierarchy mask and LoRA

In [11]:
!pip install -q --upgrade --no-deps "torchao>=0.17"
import importlib.metadata as metadata
print("torchao", metadata.version("torchao"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 31.9 MB/s eta 0:00:00
torchao 0.18.0


In [12]:
from transformers import AutoTokenizer
from src.model import GrieveAIClassifier, load_taxonomy, MURIL_CHECKPOINT
model_taxonomy = load_taxonomy(TAXONOMY_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = GrieveAIClassifier(7, 28).to("cuda").eval()
batch = tokenizer("Synthetic example: campus Wi-Fi is unavailable.", return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
with torch.no_grad():
    output = model(**{k: v.to("cuda") for k, v in batch.items()})
assert output["category_logits"].shape == (1, 7)
assert output["subcategory_logits"].shape == (1, 28)
cat = int(output["category_logits"].argmax(-1)[0])
masked = model.mask_subcategory_logits(output["subcategory_logits"][0], cat, model_taxonomy)
allowed = model_taxonomy["category_to_subcat_indices"][model_taxonomy["categories"][cat]]
assert all(torch.isneginf(masked[i]) for i in range(28) if i not in allowed)
print("Forward and parent mask passed; trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Forward and parent mask passed; trainable parameters: 322596


## 7. Run the one-epoch pipeline diagnostic
This checks training, checkpoint writing and reload. Its metrics are diagnostic only.

In [13]:
!python src/train.py --data data/processed/grievances_synthetic.csv --taxonomy config/taxonomy.json --smoke_test --batch_size 8 --seed 42 --max_length 128 --output_dir /content/GrieveAI/checkpoints/diagnostic_1_epoch

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Device: cuda | Train: 801 | Val: 275 | Test: 247
Loading weights: 100% 199/199 [00:00<00:00, 16744.23it/s]
[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  |

## 8. Inspect diagnostic metrics and explicitly reload the diagnostic checkpoint

In [14]:
import json
from src.model import MuRILInference
metrics = json.loads((Path(SMOKE_OUTPUT_DIR) / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps({"validation": metrics["validation"], "test": metrics["test"], "split": metrics["split"]}, indent=2))
reloaded = MuRILInference(SMOKE_OUTPUT_DIR, TAXONOMY_PATH, device="cuda")
print(reloaded.predict("Synthetic example: campus Wi-Fi is unavailable."))


{
  "validation": {
    "dataset_kind": "synthetic_demo_pipeline_validation",
    "category_macro_f1": 0.03940886699507389,
    "subcategory_macro_f1": 0.0015816528272044287,
    "subcategory_macro_f1_given_gold_category": 0.08149455988829447,
    "priority_mae": 2.404762672402642,
    "priority_recall_high": 0.0,
    "routing_accuracy": null,
    "routing_accuracy_note": "Requires validated gold department labels.",
    "override_rate": null,
    "override_rate_note": "Requires observed eligible human routing decisions.",
    "research_claim": "not_vcet_pilot_results"
  },
  "test": {
    "dataset_kind": "synthetic_demo_pipeline_validation",
    "category_macro_f1": 0.030020703933747412,
    "subcategory_macro_f1": 0.001225865767698437,
    "subcategory_macro_f1_given_gold_category": 0.06490916848823376,
    "priority_mae": 2.4026944993237254,
    "priority_recall_high": 0.0,
    "routing_accuracy": null,
    "routing_accuracy_note": "Requires validated gold department labels.",
    "

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'category': 'Examinations', 'subcategory': 'marks_discrepancy', 'priority': 1, 'priority_raw': 0.362, 'confidence': 0.15, 'subcategory_confidence': 0.266}


## 9. Save the reproducibility manifest

In [15]:
experiment_manifest = {
    "experiment_name": "muril_lora_synthetic_seed42_diagnostic",
    "git_commit": GIT_COMMIT,
    "dataset_path": DATA_PATH,
    "dataset_sha256": source_manifest[DATA_PATH],
    "taxonomy_path": TAXONOMY_PATH,
    "taxonomy_sha256": source_manifest[TAXONOMY_PATH],
    "random_seed": SEED,
    "split_sizes": [len(train_df), len(val_df), len(test_df)],
    "model_name": MODEL_NAME, "tokenizer_name": MODEL_NAME,
    "lora": {"r": 8, "alpha": 16, "dropout": 0.1, "target_modules": ["query", "value"]},
    "batch_size": BATCH_SIZE, "learning_rate": 2e-4, "max_sequence_length": MAX_LENGTH,
    "optimizer": "AdamW", "scheduler": "none",
    "loss": {"category": "cross_entropy (focal gamma 0)", "subcategory": "gold-parent masked cross_entropy (focal gamma 0)", "priority": "Huber delta 1, weight 0.5"},
    "epochs": 1, "metrics": metrics, "packages": versions, "source_sha256": source_manifest,
    "data_scope": "synthetic only; not VCET performance"
}
manifest_path = Path(SMOKE_OUTPUT_DIR) / "experiment_manifest.json"
manifest_path.write_text(json.dumps(experiment_manifest, indent=2), encoding="utf-8")
print("Saved manifest:", manifest_path)


Saved manifest: /content/GrieveAI/checkpoints/diagnostic_1_epoch/experiment_manifest.json


In [16]:
from collections import Counter
from sklearn.metrics import f1_score, mean_absolute_error, recall_score
from torch.nn import functional as F

def probe_predictions(frame, name):
    true_cat, pred_cat, true_sub, pred_sub, pred_sub_oracle, true_pri, pred_pri = [], [], [], [], [], [], []
    reloaded.encoder.eval()
    for start in range(0, len(frame), 32):
        part = frame.iloc[start:start+32]
        encoded = reloaded.tokenizer(part.text.astype(str).tolist(), truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
        encoded = {k: v.to("cuda") for k, v in encoded.items()}
        with torch.no_grad():
            out = reloaded.encoder(**encoded)
            pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
            cat_logits = reloaded.category_head(pooled)
            sub_logits = reloaded.subcategory_head(pooled)
            pri = reloaded.priority_head(pooled).squeeze(-1)
            cats = cat_logits.argmax(-1)
            for i in range(len(part)):
                pcat = int(cats[i])
                valid = model_taxonomy["category_to_subcat_indices"][model_taxonomy["categories"][pcat]]
                pred_sub.append(int(sub_logits[i].masked_fill(torch.tensor([j not in valid for j in range(28)], device="cuda"), float("-inf")).argmax()))
                gcat = model_taxonomy["categories"].index(part.iloc[i].category)
                gold_valid = model_taxonomy["category_to_subcat_indices"][part.iloc[i].category]
                pred_sub_oracle.append(int(sub_logits[i].masked_fill(torch.tensor([j not in gold_valid for j in range(28)], device="cuda"), float("-inf")).argmax()))
                true_cat.append(gcat)
                true_sub.append(model_taxonomy["subcategories"].index(part.iloc[i].subcategory))
            pred_cat.extend(cats.cpu().tolist())
            true_pri.extend(part.priority.astype(float).tolist())
            pred_pri.extend(pri.cpu().tolist())
    result = {
        "split": name,
        "category_macro_f1": f1_score(true_cat, pred_cat, average="macro"),
        "subcategory_macro_f1": f1_score(true_sub, pred_sub, average="macro"),
        "subcategory_macro_f1_gold_parent": f1_score(true_sub, pred_sub_oracle, average="macro"),
        "priority_mae": mean_absolute_error(true_pri, pred_pri),
        "high_priority_recall": recall_score([x >= 4 for x in true_pri], [x >= 4 for x in pred_pri], zero_division=0),
        "category_prediction_counts": dict(Counter(model_taxonomy["categories"][i] for i in pred_cat)),
    }
    print(result)
    return result

probe_predictions(train_df, "train")
probe_predictions(val_df, "validation")
probe_predictions(test_df, "test")

# Check that the saved adapter and task heads still carry gradient paths. No update is applied.
probe_texts = train_df.text.astype(str).iloc[:BATCH_SIZE].tolist()
probe_rows = train_df.iloc[:BATCH_SIZE]
encoded = reloaded.tokenizer(probe_texts, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
encoded = {k: v.to("cuda") for k, v in encoded.items()}
cat_y = torch.tensor([model_taxonomy["categories"].index(x) for x in probe_rows.category], device="cuda")
sub_y = torch.tensor([model_taxonomy["subcategories"].index(x) for x in probe_rows.subcategory], device="cuda")
pri_y = torch.tensor(probe_rows.priority.astype(float).tolist(), device="cuda")
for n, p in reloaded.encoder.named_parameters():
    p.requires_grad_("lora_" in n)
for head in (reloaded.category_head, reloaded.subcategory_head, reloaded.priority_head):
    for p in head.parameters(): p.requires_grad_(True)
reloaded.encoder.train()
encoded_out = reloaded.encoder(**encoded)
pooled = encoded_out.pooler_output if getattr(encoded_out, "pooler_output", None) is not None else encoded_out.last_hidden_state[:, 0]
cat_logits = reloaded.category_head(pooled)
sub_logits = reloaded.subcategory_head(pooled)
pri = reloaded.priority_head(pooled).squeeze(-1)
masked_gold = torch.stack([model.mask_subcategory_logits(sub_logits[i], int(cat_y[i]), model_taxonomy) for i in range(len(cat_y))])
probe_loss = F.cross_entropy(cat_logits, cat_y) + F.cross_entropy(masked_gold, sub_y) + .5 * F.huber_loss(pri, pri_y)
probe_loss.backward()
lora_norm = sum(float(p.grad.float().norm().item()) for n,p in reloaded.encoder.named_parameters() if "lora_" in n and p.grad is not None)
head_norm = sum(float(p.grad.float().norm().item()) for head in (reloaded.category_head, reloaded.subcategory_head, reloaded.priority_head) for p in head.parameters() if p.grad is not None)
print({"probe_loss": float(probe_loss.item()), "one_batch_lora_gradient_norm": lora_norm, "one_batch_head_gradient_norm": head_norm, "lora_gradient_tensors": sum(p.grad is not None for n,p in reloaded.encoder.named_parameters() if "lora_" in n)})
reloaded.encoder.zero_grad(set_to_none=True)
reloaded.encoder.eval()

{'split': 'train', 'category_macro_f1': 0.03722661703117729, 'subcategory_macro_f1': 0.0033973161202649907, 'subcategory_macro_f1_gold_parent': 0.1138246735583301, 'priority_mae': 2.3977763500627356, 'high_priority_recall': 0.0, 'category_prediction_counts': {'Examinations': 801}}
{'split': 'validation', 'category_macro_f1': 0.03940886699507389, 'subcategory_macro_f1': 0.0015816528272044287, 'subcategory_macro_f1_gold_parent': 0.08149455988829447, 'priority_mae': 2.4047626732696186, 'high_priority_recall': 0.0, 'category_prediction_counts': {'Examinations': 275}}
{'split': 'test', 'category_macro_f1': 0.030020703933747412, 'subcategory_macro_f1': 0.001225865767698437, 'subcategory_macro_f1_gold_parent': 0.06490916848823376, 'priority_mae': 2.4026944995650394, 'high_priority_recall': 0.0, 'category_prediction_counts': {'Examinations': 247}}
{'probe_loss': 4.043262958526611, 'one_batch_lora_gradient_norm': 0.4845005776005564, 'one_batch_head_gradient_norm': 1.5388873666524887, 'lora_grad

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(197285, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
                

In [17]:
from src.train import seed_everything, GrievanceDataset, evaluate
from torch.utils.data import DataLoader
from src.model import FocalLoss
import torch

seed_everything(SEED)
trace_train_ds = GrievanceDataset(train_df, tokenizer, model_taxonomy, MAX_LENGTH)
trace_val_ds = GrievanceDataset(val_df, tokenizer, model_taxonomy, MAX_LENGTH)
trace_loader = DataLoader(trace_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
trace_val_loader = DataLoader(trace_val_ds, batch_size=BATCH_SIZE, num_workers=0)
trace_model = GrieveAIClassifier(7, 28).to("cuda")
trace_params = [p for p in trace_model.parameters() if p.requires_grad]
trace_opt = torch.optim.AdamW(trace_params, lr=2e-4, weight_decay=0.01)
trace_scaler = torch.cuda.amp.GradScaler(enabled=True)
trace_cat_loss, trace_sub_loss, trace_pri_loss = FocalLoss(gamma=0.0), FocalLoss(gamma=0.0), torch.nn.HuberLoss(delta=1.0)
trace_log, amp_skips, nonfinite_grad_batches, clipped_norms = [], 0, 0, []
trace_model.train()
for step, batch in enumerate(trace_loader, 1):
    ids, mask = batch["input_ids"].cuda(), batch["attention_mask"].cuda()
    cat_y, sub_y, pri_y = batch["category_label"].cuda(), batch["subcategory_label"].cuda(), batch["priority_label"].cuda()
    token_types = batch.get("token_type_ids")
    if token_types is not None: token_types = token_types.cuda()
    trace_opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", enabled=True):
        out = trace_model(ids, mask, token_type_ids=token_types)
        masked = torch.stack([trace_model.mask_subcategory_logits(out["subcategory_logits"][i], int(cat_y[i]), model_taxonomy) for i in range(len(cat_y))])
        lc = trace_cat_loss(out["category_logits"], cat_y)
        ls = trace_sub_loss(masked, sub_y)
        lp = .5 * trace_pri_loss(out["priority_pred"], pri_y)
        loss = lc + ls + lp
    trace_scaler.scale(loss).backward()
    trace_scaler.unscale_(trace_opt)
    finite = all(p.grad is None or torch.isfinite(p.grad).all().item() for p in trace_params)
    if not finite: nonfinite_grad_batches += 1
    norm = torch.nn.utils.clip_grad_norm_(trace_params, 1.0)
    clipped_norms.append(float(norm.item()))
    scale_before = trace_scaler.get_scale()
    trace_scaler.step(trace_opt)
    trace_scaler.update()
    if trace_scaler.get_scale() < scale_before: amp_skips += 1
    trace_log.append((float(loss.item()), float(lc.item()), float(ls.item()), float(lp.item()), float(trace_scaler.get_scale())))
    if step in {1, 10, 25, 50, 75, len(trace_loader)}:
        print({"step": step, "total": trace_log[-1][0], "category": trace_log[-1][1], "masked_subcategory": trace_log[-1][2], "priority_weighted": trace_log[-1][3], "grad_scaler": trace_log[-1][4], "preclip_grad_norm": clipped_norms[-1]})
print({"steps": len(trace_log), "initial_mean_10": [sum(x[i] for x in trace_log[:10])/10 for i in range(4)], "final_mean_10": [sum(x[i] for x in trace_log[-10:])/10 for i in range(4)], "amp_skipped_steps": amp_skips, "batches_with_nonfinite_gradients": nonfinite_grad_batches, "initial_scale": trace_log[0][4], "final_scale": trace_log[-1][4], "max_preclip_grad_norm": max(clipped_norms)})
print("post_epoch_train", evaluate(trace_model, DataLoader(trace_train_ds, batch_size=32), torch.device("cuda"), model_taxonomy))
print("post_epoch_val", evaluate(trace_model, trace_val_loader, torch.device("cuda"), model_taxonomy))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_765/3781562062.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.Grad

{'step': 1, 'total': 4.397265434265137, 'category': 1.96533203125, 'masked_subcategory': 1.3795166015625, 'priority_weighted': 1.0524168014526367, 'grad_scaler': 65536.0, 'preclip_grad_norm': 0.7306807637214661}
{'step': 10, 'total': 4.392997741699219, 'category': 1.955078125, 'masked_subcategory': 1.3924560546875, 'priority_weighted': 1.0454635620117188, 'grad_scaler': 65536.0, 'preclip_grad_norm': 0.6424819231033325}
{'step': 25, 'total': 4.445065021514893, 'category': 1.9556884765625, 'masked_subcategory': 1.3941650390625, 'priority_weighted': 1.0952115058898926, 'grad_scaler': 65536.0, 'preclip_grad_norm': 0.6592843532562256}
{'step': 50, 'total': 4.20966911315918, 'category': 1.9415283203125, 'masked_subcategory': 1.3826904296875, 'priority_weighted': 0.8854506015777588, 'grad_scaler': 65536.0, 'preclip_grad_norm': 0.7926700115203857}
{'step': 75, 'total': 4.479648590087891, 'category': 1.939208984375, 'masked_subcategory': 1.3983154296875, 'priority_weighted': 1.1421244144439697,

In [18]:
seed_everything(SEED)
fp32_ds = GrievanceDataset(train_df, tokenizer, model_taxonomy, MAX_LENGTH)
fp32_train_loader = DataLoader(fp32_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
fp32_val_loader = DataLoader(GrievanceDataset(val_df, tokenizer, model_taxonomy, MAX_LENGTH), batch_size=32, num_workers=0)
fp32_model = GrieveAIClassifier(7, 28).to("cuda")
fp32_opt = torch.optim.AdamW([p for p in fp32_model.parameters() if p.requires_grad], lr=2e-4, weight_decay=0.01)
fp32_loss_fn_cat, fp32_loss_fn_sub, fp32_loss_fn_pri = FocalLoss(gamma=0.0), FocalLoss(gamma=0.0), torch.nn.HuberLoss(delta=1.0)
fp32_losses, fp32_nonfinite = [], 0
fp32_model.train()
for step, batch in enumerate(fp32_train_loader, 1):
    ids, mask = batch["input_ids"].cuda(), batch["attention_mask"].cuda()
    cat_y, sub_y, pri_y = batch["category_label"].cuda(), batch["subcategory_label"].cuda(), batch["priority_label"].cuda()
    token_types = batch.get("token_type_ids")
    if token_types is not None: token_types = token_types.cuda()
    fp32_opt.zero_grad(set_to_none=True)
    out = fp32_model(ids, mask, token_type_ids=token_types)
    masked = torch.stack([fp32_model.mask_subcategory_logits(out["subcategory_logits"][i], int(cat_y[i]), model_taxonomy) for i in range(len(cat_y))])
    lc = fp32_loss_fn_cat(out["category_logits"], cat_y)
    ls = fp32_loss_fn_sub(masked, sub_y)
    lp = .5 * fp32_loss_fn_pri(out["priority_pred"], pri_y)
    loss = lc + ls + lp
    loss.backward()
    finite = all(p.grad is None or torch.isfinite(p.grad).all().item() for p in fp32_model.parameters() if p.requires_grad)
    if not finite: fp32_nonfinite += 1
    torch.nn.utils.clip_grad_norm_(fp32_model.parameters(), 1.0)
    fp32_opt.step()
    fp32_losses.append((float(loss.item()), float(lc.item()), float(ls.item()), float(lp.item())))
    if step in {1,10,25,50,75,len(fp32_train_loader)}:
        print({"step":step,"total":fp32_losses[-1][0],"category":fp32_losses[-1][1],"subcategory":fp32_losses[-1][2],"priority_weighted":fp32_losses[-1][3]})
print({"steps":len(fp32_losses),"initial_mean_10":[sum(x[i] for x in fp32_losses[:10])/10 for i in range(4)],"final_mean_10":[sum(x[i] for x in fp32_losses[-10:])/10 for i in range(4)],"nonfinite_gradient_batches":fp32_nonfinite,"train_metrics":evaluate(fp32_model,DataLoader(fp32_ds,batch_size=32),torch.device("cuda"),model_taxonomy),"validation_metrics":evaluate(fp32_model,fp32_val_loader,torch.device("cuda"),model_taxonomy)})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'step': 1, 'total': 4.395966529846191, 'category': 1.966348648071289, 'subcategory': 1.3770949840545654, 'priority_weighted': 1.0525226593017578}
{'step': 10, 'total': 4.392526149749756, 'category': 1.9544271230697632, 'subcategory': 1.3926670551300049, 'priority_weighted': 1.045432209968567}
{'step': 25, 'total': 4.444963455200195, 'category': 1.9555141925811768, 'subcategory': 1.3935657739639282, 'priority_weighted': 1.0958832502365112}
{'step': 50, 'total': 4.209911346435547, 'category': 1.94120454788208, 'subcategory': 1.3833287954330444, 'priority_weighted': 0.8853780031204224}
{'step': 75, 'total': 4.4778642654418945, 'category': 1.9384493827819824, 'subcategory': 1.3980226516723633, 'priority_weighted': 1.1413921117782593}
{'step': 101, 'total': 5.319247245788574, 'category': 1.9300490617752075, 'subcategory': 1.3120173215866089, 'priority_weighted': 2.0771806240081787}
{'steps': 101, 'initial_mean_10': [4.403610944747925, 1.9456981897354126, 1.3839657783508301, 1.0739469051361

In [19]:
seed_everything(SEED)
headlr_ds = GrievanceDataset(train_df, tokenizer, model_taxonomy, MAX_LENGTH)
headlr_train_loader = DataLoader(headlr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
headlr_val_loader = DataLoader(GrievanceDataset(val_df, tokenizer, model_taxonomy, MAX_LENGTH), batch_size=32, num_workers=0)
headlr_model = GrieveAIClassifier(7, 28).to("cuda")
head_params = list(headlr_model.category_head.parameters()) + list(headlr_model.subcategory_head.parameters()) + list(headlr_model.priority_head.parameters())
head_param_ids = {id(p) for p in head_params}
encoder_params = [p for p in headlr_model.parameters() if p.requires_grad and id(p) not in head_param_ids]
headlr_opt = torch.optim.AdamW([{"params": encoder_params, "lr": 2e-4}, {"params": head_params, "lr": 1e-3}], weight_decay=0.01)
headlr_cat, headlr_sub, headlr_pri = FocalLoss(gamma=0.0), FocalLoss(gamma=0.0), torch.nn.HuberLoss(delta=1.0)
headlr_losses=[]
headlr_model.train()
for step,batch in enumerate(headlr_train_loader,1):
    ids,mask=batch["input_ids"].cuda(),batch["attention_mask"].cuda()
    cy,sy,py=batch["category_label"].cuda(),batch["subcategory_label"].cuda(),batch["priority_label"].cuda()
    tt=batch.get("token_type_ids")
    if tt is not None: tt=tt.cuda()
    headlr_opt.zero_grad(set_to_none=True)
    out=headlr_model(ids,mask,token_type_ids=tt)
    masked=torch.stack([headlr_model.mask_subcategory_logits(out["subcategory_logits"][i],int(cy[i]),model_taxonomy) for i in range(len(cy))])
    lc=headlr_cat(out["category_logits"],cy); ls=headlr_sub(masked,sy); lp=.5*headlr_pri(out["priority_pred"],py); loss=lc+ls+lp
    loss.backward(); torch.nn.utils.clip_grad_norm_(headlr_model.parameters(),1.0); headlr_opt.step()
    headlr_losses.append((float(loss.item()),float(lc.item()),float(ls.item()),float(lp.item())))
    if step in {1,10,25,50,75,len(headlr_train_loader)}: print({"step":step,"losses":headlr_losses[-1]})
print({"head_lr":1e-3,"lora_lr":2e-4,"steps":len(headlr_losses),"initial_mean_10":[sum(x[i] for x in headlr_losses[:10])/10 for i in range(4)],"final_mean_10":[sum(x[i] for x in headlr_losses[-10:])/10 for i in range(4)],"train_metrics":evaluate(headlr_model,DataLoader(headlr_ds,batch_size=32),torch.device("cuda"),model_taxonomy),"validation_metrics":evaluate(headlr_model,headlr_val_loader,torch.device("cuda"),model_taxonomy)})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'step': 1, 'losses': (4.395966529846191, 1.966348648071289, 1.3770949840545654, 1.0525226593017578)}
{'step': 10, 'losses': (4.366432189941406, 1.950637936592102, 1.3989489078521729, 1.0168452262878418)}
{'step': 25, 'losses': (4.3883514404296875, 1.9528437852859497, 1.4143601655960083, 1.0211472511291504)}
{'step': 50, 'losses': (4.022200584411621, 1.9170618057250977, 1.3750821352005005, 0.730056643486023)}
{'step': 75, 'losses': (4.117402076721191, 1.9154293537139893, 1.4152019023895264, 0.7867707014083862)}
{'step': 101, 'losses': (4.564540386199951, 1.9376921653747559, 1.146661639213562, 1.4801865816116333)}
{'head_lr': 0.001, 'lora_lr': 0.0002, 'steps': 101, 'initial_mean_10': [4.393534183502197, 1.9482247948646545, 1.385526967048645, 1.0597823500633239], 'final_mean_10': [3.8915228843688965, 1.9494478106498718, 1.3569637060165405, 0.5851113438606262], 'train_metrics': {'dataset_kind': 'training_dataset_metrics_require_data_provenance_review', 'category_macro_f1': 0.0388349514563

In [20]:
seed_everything(SEED)
ref_model = GrieveAIClassifier(7,28).to("cuda").eval()
headlr_model.eval()
def update_stats(module_name, trained, initial):
    trained_params=dict(trained.named_parameters()); initial_params=dict(initial.named_parameters());
    names=[n for n in trained_params if ("category_head" in n if module_name=="category_head" else ("subcategory_head" in n if module_name=="subcategory_head" else "lora_" in n))]
    delta_sq=base_sq=0.0
    for n in names:
        delta_sq += float((trained_params[n].detach().float()-initial_params[n].detach().float()).pow(2).sum().item())
        base_sq += float(initial_params[n].detach().float().pow(2).sum().item())
    print({"parameter_group":module_name,"tensors":len(names),"relative_update_norm":(delta_sq**.5)/(base_sq**.5+1e-12)})
update_stats("category_head",headlr_model,ref_model)
update_stats("subcategory_head",headlr_model,ref_model)
update_stats("lora",headlr_model,ref_model)
check = train_df.iloc[:128]
for label, probe_model in [("fresh",ref_model),("headlr_trained",headlr_model)]:
    ys=[]; ps=[]; losses=[]
    for start in range(0,len(check),32):
        part=check.iloc[start:start+32]
        enc=tokenizer(part.text.astype(str).tolist(),truncation=True,padding="max_length",max_length=MAX_LENGTH,return_tensors="pt"); enc={k:v.cuda() for k,v in enc.items()}
        with torch.no_grad():
            out=probe_model(**enc); cats=out["category_logits"]; target=torch.tensor([model_taxonomy["categories"].index(x) for x in part.category],device="cuda")
            losses.append(float(F.cross_entropy(cats,target).item())); ys.extend(target.cpu().tolist()); ps.extend(cats.argmax(-1).cpu().tolist())
    print({"model":label,"sample_category_ce":sum(losses)/len(losses),"sample_category_accuracy":sum(a==b for a,b in zip(ys,ps))/len(ys),"predictions":dict(Counter(model_taxonomy["categories"][i] for i in ps))})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'parameter_group': 'category_head', 'tensors': 4, 'relative_update_norm': 0.666336306125507}
{'parameter_group': 'subcategory_head', 'tensors': 2, 'relative_update_norm': 0.7012248514321733}
{'parameter_group': 'lora', 'tensors': 48, 'relative_update_norm': 0.5980383656611514}
{'model': 'fresh', 'sample_category_ce': 1.9452301859855652, 'sample_category_accuracy': 0.15625, 'predictions': {'Examinations': 128}}
{'model': 'headlr_trained', 'sample_category_ce': 1.9419270157814026, 'sample_category_accuracy': 0.125, 'predictions': {'Canteen': 128}}


In [21]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def collect_frozen(frame):
    pooled_parts=[]; cls_parts=[]
    ref_model.encoder.eval()
    for start in range(0,len(frame),32):
        part=frame.iloc[start:start+32]
        enc=tokenizer(part.text.astype(str).tolist(),truncation=True,padding="max_length",max_length=MAX_LENGTH,return_tensors="pt"); enc={k:v.cuda() for k,v in enc.items()}
        with torch.no_grad():
            out=ref_model.encoder(**enc)
            pooled=out.pooler_output
            cls=out.last_hidden_state[:,0]
            pooled_parts.append(pooled.float().cpu().numpy()); cls_parts.append(cls.float().cpu().numpy())
    return np.concatenate(pooled_parts),np.concatenate(cls_parts)

frozen={name:collect_frozen(frame) for name,frame in [("train",train_df),("validation",val_df),("test",test_df)]}
ytrain=np.array([model_taxonomy["categories"].index(x) for x in train_df.category])
for rep_idx,rep_name in [(0,"pooler"),(1,"cls")]:
    probe=LogisticRegression(max_iter=500,random_state=SEED,C=1.0)
    probe.fit(frozen["train"][rep_idx],ytrain)
    for split,frame in [("validation",val_df),("test",test_df)]:
        truth=np.array([model_taxonomy["categories"].index(x) for x in frame.category])
        pred=probe.predict(frozen[split][rep_idx])
        print({"representation":rep_name,"split":split,"category_macro_f1":f1_score(truth,pred,average="macro"),"accuracy":float((truth==pred).mean()),"features_std":float(frozen[split][rep_idx].std())})

{'representation': 'pooler', 'split': 'validation', 'category_macro_f1': 0.030612244897959183, 'accuracy': 0.12, 'features_std': 0.01102597825229168}
{'representation': 'pooler', 'split': 'test', 'category_macro_f1': 0.02336696760488582, 'accuracy': 0.08906882591093117, 'features_std': 0.011026632972061634}
{'representation': 'cls', 'split': 'validation', 'category_macro_f1': 0.030612244897959183, 'accuracy': 0.12, 'features_std': 0.022518306970596313}
{'representation': 'cls', 'split': 'test', 'category_macro_f1': 0.02336696760488582, 'accuracy': 0.08906882591093117, 'features_std': 0.022519635036587715}


In [22]:
def collect_mean(frame):
    parts=[]; ref_model.encoder.eval()
    for start in range(0,len(frame),32):
        part=frame.iloc[start:start+32]
        enc=tokenizer(part.text.astype(str).tolist(),truncation=True,padding="max_length",max_length=MAX_LENGTH,return_tensors="pt"); enc={k:v.cuda() for k,v in enc.items()}
        with torch.no_grad():
            out=ref_model.encoder(**enc)
            mask=enc["attention_mask"].unsqueeze(-1).float()
            parts.append(((out.last_hidden_state*mask).sum(1)/mask.sum(1).clamp(min=1)).float().cpu().numpy())
    return np.concatenate(parts)
mean_features={name:collect_mean(frame) for name,frame in [("train",train_df),("validation",val_df),("test",test_df)]}
mean_probe=LogisticRegression(max_iter=500,random_state=SEED,C=1.0).fit(mean_features["train"],ytrain)
for split,frame in [("validation",val_df),("test",test_df)]:
    truth=np.array([model_taxonomy["categories"].index(x) for x in frame.category]); pred=mean_probe.predict(mean_features[split])
    print({"representation":"masked_mean","split":split,"category_macro_f1":f1_score(truth,pred,average="macro"),"accuracy":float((truth==pred).mean()),"features_std":float(mean_features[split].std())})

{'representation': 'masked_mean', 'split': 'validation', 'category_macro_f1': 0.14804310833806014, 'accuracy': 0.23636363636363636, 'features_std': 0.022525185719132423}
{'representation': 'masked_mean', 'split': 'test', 'category_macro_f1': 0.1304029304029304, 'accuracy': 0.21862348178137653, 'features_std': 0.022542273625731468}


In [23]:
from pathlib import Path
model_file=Path("src/model.py")
source=model_file.read_text(encoding="utf-8")
old='''        if getattr(outputs, "pooler_output", None) is not None:
            pooled = outputs.pooler_output
        else:
            last_hidden = outputs.last_hidden_state
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
'''
new='''        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(dtype=last_hidden.dtype)
        pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
'''
assert old in source
source=source.replace(old,new,1)
old='''        pooled = outputs.pooler_output if getattr(outputs, "pooler_output", None) is not None else outputs.last_hidden_state[:, 0]
'''
new='''        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(dtype=last_hidden.dtype)
        pooled = (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
'''
assert old in source
model_file.write_text(source.replace(old,new,1),encoding="utf-8")
print("runtime model patch applied; source commit remains", GIT_COMMIT)
!python src/train.py --data data/processed/grievances_synthetic.csv --taxonomy config/taxonomy.json --smoke_test --batch_size 8 --seed 42 --max_length 128 --output_dir /content/GrieveAI/checkpoints/ablation_mean_pooling

runtime model patch applied; source commit remains f67a1947da6d4b749e0e041150ceb962916c55af
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Device: cuda | Train: 801 | Val: 275 | Test: 247
Loading weights: 100% 199/199 [00:00<00:00, 15092.06it/s]
[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.prediction

## Stop here

The controlled three-epoch MuRIL run is intentionally not included. Stop after the mapping checks, baseline, forward check and one-epoch diagnostic. Run three epochs only after a separate explicit instruction.